# Customer 360 Search - Cortex Search Demo

**Purpose**: Demonstrate Cortex Search for natural language customer lookup

This notebook showcases:
- Semantic search across 686K customer records
- Natural language queries with fuzzy matching
- Customer profile retrieval with meter assignments
- Integration with Cortex Agent for conversational queries

**Cortex Services Used**:
- `CUSTOMER_SEARCH_SERVICE` - Semantic search on customer data
- `AMI_METADATA_SEARCH` - Search by meter/transformer context

---

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.cortex import search
import pandas as pd

session = get_active_session()
session.use_database("FLUX_DB")
session.use_schema("PRODUCTION")

# Verify search service exists
services = session.sql("SHOW CORTEX SEARCH SERVICES IN SCHEMA PRODUCTION").to_pandas()
print("Available Search Services:")
for _, row in services.iterrows():
    print(f"  ✓ {row['name']}")

## 1. Basic Customer Search

Search by name, address, or other attributes

In [ ]:
# Search for customers by name
search_query = "residential customer in Houston Heights"

results = session.sql(f"""
    SELECT 
        CUSTOMER_ID,
        FULL_NAME,
        SERVICE_ADDRESS,
        CITY,
        CUSTOMER_SEGMENT,
        PRIMARY_METER_ID
    FROM TABLE(
        CORTEX_SEARCH(
            SERVICE => 'CUSTOMER_SEARCH_SERVICE',
            QUERY => '{search_query}',
            LIMIT => 10
        )
    )
""").to_pandas()

print(f"Search: '{search_query}'")
print(f"Results: {len(results)}")
results

## 2. Fuzzy Name Matching

Handle misspellings and partial names

In [ ]:
# Fuzzy search - handles typos
fuzzy_queries = [
    "John Smth on Oak Street",      # Typo in 'Smith'
    "commercial account downtown",  # Segment + location
    "apartment complex Galleria",   # Property type + area
]

for query in fuzzy_queries:
    results = session.sql(f"""
        SELECT CUSTOMER_ID, FULL_NAME, SERVICE_ADDRESS, CUSTOMER_SEGMENT
        FROM TABLE(
            CORTEX_SEARCH(
                SERVICE => 'CUSTOMER_SEARCH_SERVICE',
                QUERY => '{query}',
                LIMIT => 3
            )
        )
    """).to_pandas()
    
    print(f"\nQuery: '{query}'")
    print(results.to_string(index=False))

## 3. Search with Filters

Combine semantic search with structured filters

In [ ]:
# Search with attribute filters
filtered_results = session.sql("""
    SELECT 
        CUSTOMER_ID,
        FULL_NAME,
        SERVICE_ADDRESS,
        CITY,
        CUSTOMER_SEGMENT,
        ACCOUNT_STATUS
    FROM TABLE(
        CORTEX_SEARCH(
            SERVICE => 'CUSTOMER_SEARCH_SERVICE',
            QUERY => 'high usage residential',
            FILTER => {'CUSTOMER_SEGMENT': 'RESIDENTIAL', 'CITY': 'Houston'},
            LIMIT => 10
        )
    )
""").to_pandas()

print("Filtered search: 'high usage residential' + RESIDENTIAL segment + Houston")
filtered_results

## 4. Customer Profile Enrichment

Join search results with meter and usage data

In [ ]:
# Full customer profile with meter and usage data
profile_query = session.sql("""
    WITH search_results AS (
        SELECT CUSTOMER_ID, FULL_NAME, PRIMARY_METER_ID
        FROM TABLE(
            CORTEX_SEARCH(
                SERVICE => 'CUSTOMER_SEARCH_SERVICE',
                QUERY => 'commercial customer Memorial',
                LIMIT => 5
            )
        )
    )
    SELECT 
        sr.CUSTOMER_ID,
        sr.FULL_NAME,
        c.SERVICE_ADDRESS,
        c.CUSTOMER_SEGMENT,
        m.TRANSFORMER_ID,
        m.CIRCUIT_ID,
        ROUND(AVG(a.USAGE_KWH) * 96 * 30, 2) as EST_MONTHLY_KWH
    FROM search_results sr
    JOIN CUSTOMERS_MASTER_DATA c ON sr.CUSTOMER_ID = c.CUSTOMER_ID
    JOIN METER_INFRASTRUCTURE m ON sr.PRIMARY_METER_ID = m.METER_ID
    LEFT JOIN AMI_INTERVAL_READINGS a ON m.METER_ID = a.METER_ID 
        AND a.TIMESTAMP >= DATEADD('day', -30, CURRENT_DATE())
    GROUP BY 1, 2, 3, 4, 5, 6
""").to_pandas()

print("Enriched Customer Profiles:")
profile_query

## 5. AMI Metadata Search

Search by transformer, circuit, or substation context

In [ ]:
# Search meters by infrastructure context
ami_search = session.sql("""
    SELECT 
        METER_ID,
        TRANSFORMER_ID,
        SUBSTATION_ID,
        CITY,
        COUNTY_NAME,
        CUSTOMER_SEGMENT_ID
    FROM TABLE(
        CORTEX_SEARCH(
            SERVICE => 'AMI_METADATA_SEARCH',
            QUERY => 'meters near substation SUB-HOU-001 with high usage',
            LIMIT => 10
        )
    )
""").to_pandas()

print("AMI Metadata Search Results:")
ami_search

## Key Takeaways

1. **Semantic Understanding**: Cortex Search understands intent, not just keywords
2. **Fuzzy Matching**: Handles typos and variations automatically
3. **Hybrid Search**: Combine semantic search with structured filters
4. **Enrichment Pattern**: Join search results with operational data
5. **Multiple Services**: Different search services for different domains